# Serveur MCP `rag_multimodal`

Serveur MCP (streamable-http) qui publie l'outil `rag_multimodal`. Derrière l'outil, un agent RAG cherche des passages et des figures dans les manuels, puis produit une réponse sourcée. La photo jointe est d'abord décrite par un modèle vision.

Exécuter toutes les cellules : la dernière lance le serveur sur `http://localhost:8000/mcp` et reste active.

In [ ]:
import base64
import os
import re
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama, OllamaEmbeddings
from mcp.server.fastmcp import FastMCP

DONNEES = Path("donnees")
LLM = ChatOllama(model=os.getenv("OLLAMA_LLM", "qwen2.5:7b"), temperature=0)
VISION = ChatOllama(model=os.getenv("OLLAMA_VISION", "qwen2.5vl:7b"), temperature=0)
EMBEDDINGS = OllamaEmbeddings(model=os.getenv("OLLAMA_EMBED", "nomic-embed-text"))

FIGURE = re.compile(r"!\[(.+?)\]\((.+?)\)")
REF_IMAGE = re.compile(r"^[\w-]+$")

In [ ]:
def charger_manuels():
    """Découpe chaque manuel par section (## ...) : une section = un passage, une figure = sa légende."""
    passages, figures = [], []
    for manuel in sorted((DONNEES / "manuels").glob("*.md")):
        for section in manuel.read_text(encoding="utf-8").split("\n## ")[1:]:
            titre, _, corps = section.partition("\n")
            source = f"{manuel.stem}, §{titre.strip()}"
            for legende, chemin in FIGURE.findall(corps):
                figures.append(Document(legende, metadata={"source": source, "figure": chemin}))
            passages.append(Document(f"{titre}\n{FIGURE.sub('', corps).strip()}", metadata={"source": source}))
    return passages, figures

In [ ]:
_passages, _figures = charger_manuels()
INDEX_PASSAGES = InMemoryVectorStore.from_documents(_passages, EMBEDDINGS)
INDEX_FIGURES = InMemoryVectorStore.from_documents(_figures, EMBEDDINGS)

In [ ]:
@tool
def rechercher_passages(requete: str) -> str:
    """Recherche dans les manuels les passages de texte les plus pertinents pour la requête."""
    docs = INDEX_PASSAGES.similarity_search(requete, k=3)
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

In [ ]:
@tool
def rechercher_figures(requete: str) -> str:
    """Recherche dans les manuels les figures (pictogrammes, schémas) dont la légende correspond à la requête."""
    docs = INDEX_FIGURES.similarity_search(requete, k=2)
    return "\n".join(f"[{d.metadata['source']} | {d.metadata['figure']}] {d.page_content}" for d in docs)

In [ ]:
AGENT_RAG = create_agent(
    LLM,
    [rechercher_passages, rechercher_figures],
    system_prompt=(
        "Tu es l'agent RAG du service après-vente. Pour chaque demande, appelle rechercher_passages "
        "ET rechercher_figures. Réponds en français uniquement à partir des éléments trouvés. "
        "Cite chaque source entre crochets telle qu'elle apparaît, y compris le fichier de la figure utilisée. "
        "Si les éléments ne permettent pas de répondre, dis-le au lieu d'inventer."
    ),
)

In [ ]:
async def decrire_image(image_ref: str) -> str:
    if not REF_IMAGE.match(image_ref):
        raise ValueError(f"Référence d'image invalide : {image_ref}")
    chemin = DONNEES / "images" / f"{image_ref}.png"
    if not chemin.exists():
        raise ValueError(f"Image introuvable : {image_ref}")
    image_b64 = base64.b64encode(chemin.read_bytes()).decode()
    message = HumanMessage(content=[
        {"type": "text", "text": "Décris précisément cette photo d'écran d'appareil : modèle, voyants "
                                 "(couleur), pictogrammes et textes affichés. Décris, n'interprète pas."},
        {"type": "image_url", "image_url": f"data:image/png;base64,{image_b64}"},
    ])
    return (await VISION.ainvoke([message])).content

In [ ]:
mcp = FastMCP("sav-documentation")

In [ ]:
@mcp.tool()
async def rag_multimodal(question: str, image_ref: str | None = None) -> str:
    """Répond à une question de dépannage à partir des manuels (passages et figures), avec sources.

    question : la question du client.
    image_ref : référence de la photo d'écran jointe (ex. img_px200_01), si le client en a fourni une.
    """
    demande = question
    if image_ref:
        demande += f"\n\nDescription de la photo jointe ({image_ref}) : {await decrire_image(image_ref)}"
    resultat = await AGENT_RAG.ainvoke({"messages": [("user", demande)]})
    return resultat["messages"][-1].content

In [ ]:
# Bloque la cellule tant que le serveur tourne (Interrompre le kernel pour l'arrêter)
await mcp.run_streamable_http_async()